# 02 · PyRMD results: model performance and repurposing candidates

Run this notebook after:
1. `notebooks/01_data_curation.ipynb` (training sets and screening library)
2. `python scripts/run_pyrmd.py benchmark` (and optionally `python scripts/epsilon_scan.py --apply`)
3. `python scripts/run_pyrmd.py screening`

It summarises how well the model separates ALK5 actives from inactives, then ranks the approved drugs
that PyRMD predicts to be ALK5 inhibitors. It also writes `results/summary.md`, which holds every number
reported for this project.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")
sns.set_theme(style="whitegrid", context="notebook")

PROCESSED, RESULTS, FIGURES = Path("../data/processed"), Path("../results"), Path("../figures")
TOP_N = 20

curation = json.loads((PROCESSED / "curation_summary.json").read_text())
pd.Series(curation).to_frame("value")

## 1 · Benchmark: can the model tell ALK5 actives from inactives?
PyRMD's benchmark mode uses repeated stratified k-fold cross-validation (5 folds × 3 repeats). Each metric
is the mean with a 95 % confidence interval.

* **TPR** (recall): share of true actives recovered. **FPR**: share of inactives wrongly called active.
* **ROC AUC / PRC AUC**: ranking quality across all thresholds.
* **BEDROC**: early recognition, i.e. whether actives are ranked at the very top, which matters most in virtual screening.

In [ ]:
bench = pd.read_csv(RESULTS / "benchmark" / "benchmark_results.csv").iloc[-1]
metrics = ["TPR", "FPR", "Precision", "F Score", "ROC AUC", "PRC AUC", "BEDROC"]
bench_table = pd.DataFrame({"mean": [bench[m] for m in metrics],
                            "95% CI (±)": [bench[f"{m} CI"] for m in metrics]}, index=metrics).round(3)
print(f"Training set: {int(bench['Training Actives'])} actives, {int(bench['Training Inactives'])} inactives | "
      f"fingerprint: {bench['fp_type']} | epsilon cutoffs: actives {bench['epsilon_cutoff_actives']}, "
      f"inactives {bench['epsilon_cutoff_inactives']}")
bench_table

In [ ]:
for plot in ["ROC_curve.png", "PRC_curve.png"]:
    path = RESULTS / "benchmark" / plot
    if path.exists():
        display(Image(filename=str(path), width=420))

### Epsilon cutoff tuning (optional)
Shown only if `scripts/epsilon_scan.py` has been run.

In [ ]:
scan_file = RESULTS / "epsilon_scan" / "epsilon_scan_ranked.csv"
scan = None
if scan_file.exists():
    scan = pd.read_csv(scan_file)
    pivot = scan.pivot_table(index="epsilon_cutoff_actives", columns="epsilon_cutoff_inactives", values="Youden J")
    fig, ax = plt.subplots(figsize=(6, 3.8))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar_kws={"label": "Youden J (TPR − FPR)"})
    ax.set(title="Epsilon cutoff scan", xlabel="epsilon (inactives)", ylabel="epsilon (actives)")
    fig.tight_layout()
    fig.savefig(FIGURES / "02_epsilon_scan.png", dpi=200)
    plt.show()
    display(scan.round(3).head(5))
else:
    print("No epsilon scan found - skipping.")

## 2 · Screening: which approved drugs are predicted ALK5 inhibitors?
PyRMD outputs only the compounds it predicts as active, with:
* **RMD_score**: confidence (higher is better)
* **similarity / most similar compound**: Tanimoto similarity to the closest known active in the training set, a measure of novelty
* **potential_pain**: flag for pan-assay interference compounds (PAINS), which often give false positives

The shortlist removes PAINS and drugs that were already in the training set (these are known ALK5 actives,
not new predictions).

In [ ]:
pred_file = RESULTS / "screening" / "database_predictions.csv"
library = pd.read_csv(PROCESSED / "screening_library_annotated.csv")
preds = pd.read_csv(pred_file) if pred_file.exists() else pd.DataFrame()

if preds.empty:
    print("PyRMD predicted no actives in the screening library.")
    shortlist = preds
else:
    # PyRMD appends a suffix such as "-0" to each title; strip it to match the library
    preds["Title"] = preds["Title"].astype(str).str.replace(r"-\d+$", "", regex=True)
    preds = preds.merge(library[["Title", "name", "in_training_set"]], on="Title", how="left")
    preds = preds.sort_values("RMD_score", ascending=False).drop_duplicates("Title")
    shortlist = preds[(preds["potential_pain"] != "Yes") & (~preds["in_training_set"].fillna(False).astype(bool))]
    print(f"Library screened: {curation['screening_library_size']:,} compounds")
    print(f"Predicted ALK5 actives: {len(preds):,}")
    print(f"After removing PAINS and known training compounds: {len(shortlist):,}")

shortlist.head(TOP_N)[["Title", "name", "RMD_score", "similarity", "most similar compound", "potential_pain"]] if len(shortlist) else shortlist

In [ ]:
if len(preds):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    sns.histplot(preds["RMD_score"], bins=30, color="#2a9d8f", ax=axes[0])
    axes[0].set(title="Confidence of predicted actives", xlabel="RMD score")
    sns.histplot(preds["similarity"], bins=20, color="#264653", ax=axes[1])
    axes[1].set(title="Similarity to nearest known ALK5 active", xlabel="Tanimoto similarity")
    fig.tight_layout()
    fig.savefig(FIGURES / "02_screening_scores.png", dpi=200)
    plt.show()

In [ ]:
top = shortlist.head(12)
if len(top):
    legends = [f"{(n if isinstance(n, str) and n else t)[:22]}  |  RMD {s:.1f}"
               for n, t, s in zip(top["name"], top["Title"], top["RMD_score"])]
    grid = Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in top["Smiles"]], molsPerRow=4,
                                subImgSize=(280, 220), legends=legends, returnPNG=False)
    grid.save(FIGURES / "02_top_candidates.png")
    display(grid)

## 3 · Save the shortlist and the project summary

In [ ]:
cols = ["Title", "name", "Smiles", "RMD_score", "similarity", "most similar compound", "potential_pain"]
if len(shortlist):
    shortlist.head(TOP_N)[cols].to_csv(RESULTS / "top_candidates.csv", index=False)


def md_table(frame):
    header = "| " + " | ".join(frame.columns) + " |\n|" + "---|" * len(frame.columns) + "\n"
    return header + "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in frame.itertuples(index=False))


lines = [
    "# Results summary",
    "",
    "*Generated automatically by `notebooks/02_results_analysis.ipynb`.*",
    "",
    "## Data",
    f"- Target: {curation['target']}",
    f"- ChEMBL records retrieved: {curation['raw_records']:,}; after cleaning: {curation['clean_records']:,}",
    f"- Unique compounds: {curation['unique_compounds']:,}",
    f"- Training set: **{curation['actives']:,} actives** (≤ {curation['active_cutoff_nM']:,} nM) and "
    f"**{curation['inactives']:,} inactives** (≥ {curation['inactive_cutoff_nM']:,} nM); "
    f"{curation['active_scaffolds']:,} distinct active scaffolds",
    "",
    "## Model benchmark (PyRMD, 5-fold × 3 repeated stratified CV)",
    f"- Epsilon cutoffs: actives {bench['epsilon_cutoff_actives']}, inactives {bench['epsilon_cutoff_inactives']}",
    "",
    md_table(bench_table.reset_index().rename(columns={"index": "metric"})),
    "",
    "## Virtual screening",
    f"- Library screened: {curation['screening_library_size']:,} compounds",
    f"- Predicted ALK5 actives: {len(preds):,}",
    f"- Shortlist (no PAINS, not in training set): {len(shortlist):,}",
]
if len(shortlist):
    top10 = shortlist.head(10).assign(RMD_score=lambda d: d["RMD_score"].round(2),
                                      similarity=lambda d: d["similarity"].round(2))
    lines += ["", "### Top 10 candidates", "",
              md_table(top10[["Title", "name", "RMD_score", "similarity", "most similar compound"]].fillna(""))]

(RESULTS / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))